#**ML Models (CPU)**

## Importing in Data

In [87]:
#!rm -r Multicore_2026S


In [88]:
!git clone https://github.com/seernono2001/Multicore_2026S

fatal: destination path 'Multicore_2026S' already exists and is not an empty directory.


In [89]:
#Make all datasets
import pandas
#results/barrier_heavy direcotry
barrier_heavy = pandas.read_csv('Multicore_2026S/results/results_barrier_heavy.csv')
compute = pandas.read_csv('Multicore_2026S/results/results_compute.csv')
stream = pandas.read_csv('Multicore_2026S/results/results_stream.csv')

#concatenate all data
data = pandas.concat([barrier_heavy, compute, stream])
print(data)


     Index  ProblemSize  Threads  Operation  AvgSqTime  AvgParTime  AvgSpeedup
0        1         1004       22          2   0.000947    0.004295    0.220492
1        2         1019       11          2   0.000969    0.002159    0.449193
2        3         1027       40          2   0.000965    0.007539    0.128020
3        4         1032       45          2   0.000973    0.009406    0.103844
4        5         1037       60          2   0.000981    0.042648    0.025595
..     ...          ...      ...        ...        ...         ...         ...
995    996     88728965       46          1   5.901622    0.693329    8.286426
996    997     91423191       46          1   1.270303    0.516925    2.519583
997    998     92325702       45          1   1.719941    0.333705    5.079473
998    999     95281654       52          1   1.338509    0.378780    3.534867
999   1000     97283652       15          1   2.296574    0.960196    3.117948

[3000 rows x 7 columns]


In [90]:
import pandas as pd
import numpy as np

# Drop columns not used for modelling
data_use = data.copy()
data_use = data_use.drop(['Index', 'AvgSqTime', 'AvgParTime'], axis=1)

# ── Feature Engineering ──────────────────────────────────────────
# Log-scale: ProblemSize spans 1K~100M (5 orders of magnitude)
data_use['log_ProblemSize'] = np.log10(data_use['ProblemSize'])
# log2(threads) matches Amdahl's Law intuition
data_use['log_Threads']     = np.log2(data_use['Threads'])
# Work per thread: core Amdahl feature
data_use['work_per_thread'] = data_use['ProblemSize'] / data_use['Threads']
# Barrier overhead: 200 barriers / problem_size
# Large when array is small → explains speedup < 1 in barrier_heavy
data_use['barrier_overhead'] = 200.0 / data_use['ProblemSize']
# Keep ProblemSize & Threads for filtering; dropped from X below

print(data_use.head())
print(data_use.columns.tolist())


   ProblemSize  Threads  Operation  AvgSpeedup  log_ProblemSize  log_Threads  \
0         1004       22          2    0.220492         3.001734     4.459432   
1         1019       11          2    0.449193         3.008174     3.459432   
2         1027       40          2    0.128020         3.011570     5.321928   
3         1032       45          2    0.103844         3.013680     5.491853   
4         1037       60          2    0.025595         3.015779     5.906891   

   work_per_thread  barrier_overhead  
0        45.636364          0.199203  
1        92.636364          0.196271  
2        25.675000          0.194742  
3        22.933333          0.193798  
4        17.283333          0.192864  
['ProblemSize', 'Threads', 'Operation', 'AvgSpeedup', 'log_ProblemSize', 'log_Threads', 'work_per_thread', 'barrier_overhead']


## Barrier Heavy Models

In [91]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
barrier_heavy = data_use[data_use['Operation'] == 2]

#scale and split data, drop speedup and GPU
X = barrier_heavy.drop(['AvgSpeedup','Operation','ProblemSize','Threads'], axis=1)
y = barrier_heavy['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Linear Regression Baseline

In [92]:
#import linear regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#print deature importances along with their names
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])



MSE: 15.042405882811018
R2: 0.2832665234978956
log_ProblemSize : 1.6170008645845444
log_Threads : 0.6304894223567349
work_per_thread : -0.668849485805091
barrier_overhead : -1.0007501123147455


### Linear Regression (Big Problem Sizes)

In [93]:
#same thing but with problem sizes >1 mil
barrier_heavy_big = barrier_heavy[barrier_heavy['ProblemSize'] > 1000000]
print(barrier_heavy_big)
X = barrier_heavy_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads'], axis=1)
y = barrier_heavy_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4
                                                    )
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


     ProblemSize  Threads  Operation  AvgSpeedup  log_ProblemSize  \
589      1001754       51          2   16.742466         6.000761   
590      1002543       17          2   10.846576         6.001103   
591      1002780       48          2   16.569466         6.001206   
592      1011433        3          2    3.022394         6.004937   
593      1020319       12          2    9.872406         6.008736   
..           ...      ...        ...         ...              ...   
995     88728965       46          2    6.474242         7.948065   
996     91423191       46          2   14.206313         7.961056   
997     92325702       45          2   11.504578         7.965323   
998     95281654       52          2    5.905959         7.979009   
999     97283652       15          2    4.432059         7.988040   

     log_Threads  work_per_thread  barrier_overhead  
589     5.672425     1.964224e+04          0.000200  
590     4.087463     5.897312e+04          0.000199  
591     5

In [94]:
#train
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 13.25662073631157
R2: 0.3491171005357657
log_ProblemSize : 3.132241636389959
log_Threads : 1.4411807122711098
work_per_thread : -0.02343872076997755
barrier_overhead : 4.969193156474828


### Random Forests (Baseline)

In [95]:
#use the same barrier_heavy
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score


#bring back barrier_heavy
barrier_heavy = data_use[data_use['Operation'] == 2]
X = barrier_heavy.drop(['AvgSpeedup','Operation','ProblemSize','Threads'], axis=1)
y = barrier_heavy['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [96]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 6.388632419557369
R2: 0.695597448982827
log_ProblemSize : 0.20908594545540674
log_Threads : 0.12214421900763127
work_per_thread : 0.22652067670895298
barrier_overhead : 0.44224915882800897


### Random Forests (Big Problem Sizes)

In [97]:
#bring back barrier_heavy_big
barrier_heavy_big = barrier_heavy[barrier_heavy['ProblemSize'] > 1000000]
X = barrier_heavy_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads'], axis=1)
y = barrier_heavy_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [98]:
#train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 9.444115593664108
R2: 0.5727681511154352
log_ProblemSize : 0.22624079676233078
log_Threads : 0.1752628495239369
work_per_thread : 0.3556108503537579
barrier_overhead : 0.24288550335997441


## Stream Models

In [99]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
stream = data_use[data_use['Operation'] == 1]

#scale and split data, drop speedup and GPU
X = stream.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = stream['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



### Linear Regression Baseline

In [100]:
#same as barrier_heavy
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])


MSE: 1.104362067784166
R2: 0.7242094094604717
log_ProblemSize : 1.7118044551334866
log_Threads : -0.26675478669425434
work_per_thread : -0.246531611252231


### Linear Regression (Big Problem Sizes)

In [101]:

stream_big = stream[stream['ProblemSize'] > 1000000]
X = stream_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = stream_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [102]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances same as earlier
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])



MSE: 1.1509631135509428
R2: 0.328789412648663
log_ProblemSize : 0.9297069262809234
log_Threads : -0.07549531414271876
work_per_thread : -0.404539739943971


### Random Forests

In [103]:
#bring back sub data
stream = data_use[data_use['Operation'] == 1]
#print stream
print(stream)
#print barrier_heavya
print(barrier_heavy)
#save sub data as csv in direcotry
stream.to_csv('stream.csv', index=False)
# save add data
barrier_heavy.to_csv('barrier_heavy.csv', index=False)
X = stream.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = stream['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



     ProblemSize  Threads  Operation  AvgSpeedup  log_ProblemSize  \
0           1004       22          1    0.005156         3.001734   
1           1019       11          1    0.010948         3.008174   
2           1027       40          1    0.002739         3.011570   
3           1032       45          1    0.002640         3.013680   
4           1037       60          1    0.001839         3.015779   
..           ...      ...        ...         ...              ...   
995     88728965       46          1    8.286426         7.948065   
996     91423191       46          1    2.519583         7.961056   
997     92325702       45          1    5.079473         7.965323   
998     95281654       52          1    3.534867         7.979009   
999     97283652       15          1    3.117948         7.988040   

     log_Threads  work_per_thread  barrier_overhead  
0       4.459432     4.563636e+01          0.199203  
1       3.459432     9.263636e+01          0.196271  
2       5

In [104]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.7165813761625178
R2: 0.8210492675667377
log_ProblemSize : 0.17929498855422027
log_Threads : 0.06176574943554613
work_per_thread : 0.7589392620102335


### Random Forests (Big Problem Sizes)

In [105]:
#bring bad stream
stream_big = stream[stream['ProblemSize'] > 1000000]
X = stream_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = stream_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [106]:
#train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.8868793898517194
R2: 0.482795904435542
log_ProblemSize : 0.6111414447903002
log_Threads : 0.1390531211390766
work_per_thread : 0.2498054340706232


## Compute Models

In [107]:
#same pre processing as stream
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
compute = data_use[data_use['Operation'] == 0]
#scale and split data, drop gpu speedup etc
X = compute.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = compute['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)





### Linear Regression Baseline

In [108]:
#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.20968641814003797
R2: 0.8170856417782156
log_ProblemSize : 1.0255315459580836
log_Threads : -0.3490025604345642
work_per_thread : -0.233145904900311


### Linear Regression (Big Problem Sizes)



In [109]:
#make the big problem set
compute_big = compute[compute['ProblemSize'] > 1000]
X = compute_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [110]:
#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.20968641814003797
R2: 0.8170856417782156
log_ProblemSize : 1.0255315459580836
log_Threads : -0.3490025604345642
work_per_thread : -0.233145904900311


### Linear Regression (Bigger Problem Sizes)


In [111]:
#now do when problem size >1500
compute_big = compute[compute['ProblemSize'] > 1500]
X = compute_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [112]:
#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.23858152082447104
R2: 0.7933701588846727
log_ProblemSize : 1.0151306740630648
log_Threads : -0.36238683120497983
work_per_thread : -0.21588744533424337


### Random Forests

In [113]:
#Bring back mult
compute = data_use[data_use['Operation'] == 0]
X = compute.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = compute['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [114]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])


MSE: 0.02732196830392807
R2: 0.9761664091456256
log_ProblemSize : 0.027852802887392378
log_Threads : 0.041184104488529175
work_per_thread : 0.9309630926240784


### Random Forests (Big Problem Sizes)

---



In [115]:
#make bigger
compute_big = compute[compute['ProblemSize'] > 1000]
X = compute_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [116]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.02732196830392807
R2: 0.9761664091456256
log_ProblemSize : 0.027852802887392378
log_Threads : 0.041184104488529175
work_per_thread : 0.9309630926240784


### Random Forests (Bigger Problem Sizes)

In [117]:
#bigger problem size
compute_big = compute[compute['ProblemSize'] > 1500]
X = compute_big.drop(['AvgSpeedup','Operation','ProblemSize','Threads','barrier_overhead'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [118]:
#make model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.05132038852927629
R2: 0.955552619116788
log_ProblemSize : 0.02448747519936326
log_Threads : 0.04726636564481498
work_per_thread : 0.9282461591558218


## Everything Model

### Linear Regression


In [119]:
#use all data
#one hot encode the operation
data_use = pd.get_dummies(data_use, columns=['Operation'])
#scale and split data
X = data_use.drop(['AvgSpeedup','ProblemSize','Threads'], axis=1)
y = data_use['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [120]:
#build model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 5.6540517098983285
R2: 0.48221295895148786
log_ProblemSize : 1.6144736215861197
log_Threads : 0.000731505377137523
work_per_thread : -0.41558890233523643
barrier_overhead : -0.12698870132006976
Operation_0 : -0.7109214202816787
Operation_1 : -0.41980890365902923
Operation_2 : 1.1393931778596151


### Linear Regression (Big Problem Sizes)


In [121]:
#extract mult rows in data_use
compute = data_use[data_use['Operation_0'] == 1]
#cut rows with problem size less than 1000
compute_big = compute[compute['ProblemSize'] > 1000]
#extract the rest
ro_data = data_use[data_use['Operation_0'] == 0]
#cut rows with problem size less than 1 mil
ro_data_big = ro_data[ro_data['ProblemSize'] > 1000000]

data_use_big = pd.concat([compute_big, ro_data_big])
#same pre processing
X = data_use_big.drop(['AvgSpeedup','ProblemSize','Threads'], axis=1)
y = data_use_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)




In [122]:
#model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 4.420235934865877
R2: 0.4636062342000673
log_ProblemSize : 0.8664281683749637
log_Threads : 0.04921874688082695
work_per_thread : -0.36094150650823176
barrier_overhead : 0.009416731953034496
Operation_0 : -0.9076666741198228
Operation_1 : -0.13086772753163886
Operation_2 : 1.2109201217356151


### Random Forests


In [123]:
#bring back all data
X = data_use.drop(['AvgSpeedup','ProblemSize','Threads'], axis=1)
y = data_use['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [124]:
#build model
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
 print(feature_names[i], ':', feature_importances[i])

MSE: 1.3500896193397318
R2: 0.8763614227431955
log_ProblemSize : 0.17983016994008494
log_Threads : 0.08783878943214683
work_per_thread : 0.23881553527634616
barrier_overhead : 0.21711896947029133
Operation_0 : 0.004148112871074227
Operation_1 : 0.007555688337449757
Operation_2 : 0.2646927346726068


### Random Forests (Bigger Problem Size)

In [125]:
#bring back
X = data_use_big.drop(['AvgSpeedup','ProblemSize','Threads'], axis=1)
y = data_use_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [126]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
 print(feature_names[i], ':', feature_importances[i])

MSE: 2.870200939619728
R2: 0.6517023269139488
log_ProblemSize : 0.09759148999056025
log_Threads : 0.09789501705349622
work_per_thread : 0.28578698545852566
barrier_overhead : 0.125621107075156
Operation_0 : 0.04538042868182111
Operation_1 : 0.019058146557104437
Operation_2 : 0.32866682518333623
